In [19]:
%pip install osmnx geopandas geopy networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
from geopy.distance import geodesic

In [21]:
place = "Bengaluru, Karnataka, India"

tags = {
    "amenity": ["hospital", "school"],
    "shop": "mall"
}

amenities = ox.features_from_place(place, tags)

print(amenities.head())
print()
print("Total amenities downloaded:", len(amenities))

                                    geometry   amenity  \
element id                                               
node    247966855   POINT (77.6026 12.87721)    school   
        263472508  POINT (77.65183 12.91875)    school   
        267214379  POINT (77.67749 12.91342)  hospital   
        303924572   POINT (77.58575 12.9235)  hospital   
        307401529  POINT (77.67655 12.92637)  hospital   

                                                 name  \
element id                                              
node    247966855                                 NaN   
        263472508  Angel Heart Montessori Play School   
        267214379                              Clinic   
        303924572                     Shanti Hospital   
        307401529                     Apollo Hospital   

                                        name:kn healthcare  \
element id                                                   
node    247966855                           NaN        NaN   
       

In [22]:
amenities = amenities.reset_index()

# Create a single category column
amenities['category'] = amenities['amenity']
amenities.loc[amenities['shop'] == 'mall', 'category'] = 'mall'

print(amenities[['category']].value_counts())

category
school      1459
hospital    1073
mall          74
Name: count, dtype: int64


In [23]:
# Convert polygons and multipolygons into representative points
amenities['geometry'] = amenities['geometry'].representative_point()

# Extract coordinates
amenities['latitude'] = amenities.geometry.y
amenities['longitude'] = amenities.geometry.x

# Keep only the columns we need
amenities = amenities[['name', 'category', 'latitude', 'longitude']]

print(amenities.head())

                                 name  category   latitude  longitude
0                                 NaN    school  12.877212  77.602599
1  Angel Heart Montessori Play School    school  12.918750  77.651833
2                              Clinic  hospital  12.913418  77.677485
3                     Shanti Hospital  hospital  12.923504  77.585752
4                     Apollo Hospital  hospital  12.926371  77.676546


In [24]:
hospitals = amenities[amenities['category'] == 'hospital'].copy()
schools = amenities[amenities['category'] == 'school'].copy()
malls = amenities[amenities['category'] == 'mall'].copy()

print(f"Hospitals: {len(hospitals)}")
print(f"Schools:   {len(schools)}")
print(f"Malls:     {len(malls)}")

Hospitals: 1073
Schools:   1459
Malls:     74


In [25]:
amenities.to_csv('../data/processed/bangalore_amenities.csv', index=False)
hospitals.to_csv('../data/processed/bangalore_hospitals.csv', index=False)
schools.to_csv('../data/processed/bangalore_schools.csv', index=False)
malls.to_csv('../data/processed/bangalore_malls.csv', index=False)

print("Processed amenity datasets saved successfully!")

Processed amenity datasets saved successfully!


In [26]:
def nearest_distance(lat, lon, locations_df):
    """
    Returns the distance in kilometers to the nearest location
    from the provided dataframe.
    """
    distances = locations_df.apply(
        lambda row: geodesic(
            (lat, lon),
            (row['latitude'], row['longitude'])
        ).km,
        axis=1
    )
    return distances.min()

In [27]:
# MG Road area, Bengaluru
sample_lat = 12.9716
sample_lon = 77.5946

hospital_distance = nearest_distance(sample_lat, sample_lon, hospitals)
school_distance = nearest_distance(sample_lat, sample_lon, schools)
mall_distance = nearest_distance(sample_lat, sample_lon, malls)

print(f"Nearest hospital: {hospital_distance:.2f} km")
print(f"Nearest school:   {school_distance:.2f} km")
print(f"Nearest mall:     {mall_distance:.2f} km")

Nearest hospital: 0.40 km
Nearest school:   0.15 km
Nearest mall:     0.18 km


In [28]:
sample_features = {
    "distance_to_hospital_km": hospital_distance,
    "distance_to_school_km": school_distance,
    "distance_to_mall_km": mall_distance
}

print(pd.Series(sample_features))

distance_to_hospital_km    0.403804
distance_to_school_km      0.149402
distance_to_mall_km        0.182309
dtype: float64
